<a href="https://colab.research.google.com/github/keksenia/cstati-event-analytics/blob/main/notebooks/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Audit

Цель ноутбука — провести первичный аудит источников данных для проекта cstati event analytics.

На этом этапе мы проверяем:

- какие raw CSV-файлы доступны;
- сколько в них строк и колонок;
- какие поля похожи на персональные идентификаторы;
- какие поля могут отвечать за регистрацию, оплату, посещение и сегменты;
- все ли события покрыты `manual/event_metadata.csv` и `manual/event_aliases.csv`;
- какие data quality issues нужно учесть перед построением витрин.

Этот ноутбук работает с private layer. Исходные CSV с персональными данными не должны попадать в публичный GitHub.

In [78]:
from pathlib import Path

MANUAL_DIR = Path("/content/manual")
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

print(MANUAL_DIR.exists())

True


In [79]:
from pathlib import Path
import re
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("/content")
RAW_DIR = Path("/content")
INTERIM_AUDIT_DIR = PROJECT_ROOT / "data" / "interim_private" / "audit"
MANUAL_DIR = PROJECT_ROOT / "manual"

INTERIM_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("RAW_DIR exists:", RAW_DIR.exists())
print("MANUAL_DIR exists:", MANUAL_DIR.exists())

PROJECT_ROOT: /content
RAW_DIR: /content
RAW_DIR exists: True
MANUAL_DIR exists: True


In [80]:
required_paths = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / ".gitignore",
    RAW_DIR,
    PROJECT_ROOT / "data" / "interim_private",
    PROJECT_ROOT / "data" / "processed_public",
    MANUAL_DIR / "event_metadata.csv",
    MANUAL_DIR / "event_aliases.csv",
    MANUAL_DIR / "attendance_corrections.csv",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "sql",
    PROJECT_ROOT / "docs",
]

structure_check = pd.DataFrame({
    "path": [str(p.relative_to(PROJECT_ROOT)) for p in required_paths],
    "exists": [p.exists() for p in required_paths],
    "type": [
        "dir" if p.exists() and p.is_dir()
        else "file" if p.exists()
        else "missing"
        for p in required_paths
    ]
})

structure_check

,path,exists,type
0,README.md,False,missing
1,.gitignore,False,missing
2,.,True,dir
3,data/interim_private,True,dir
4,data/processed_public,False,missing
5,manual/event_metadata.csv,True,file
6,manual/event_aliases.csv,True,file
7,manual/attendance_corrections.csv,True,file
8,src,False,missing
9,sql,False,missing


In [81]:
raw_files = sorted(RAW_DIR.glob("*.csv"))

print(f"Найдено raw CSV-файлов: {len(raw_files)}")

for file in raw_files:
    print("-", file.name)

Найдено raw CSV-файлов: 20
- Копия Анализ аудитории - ANNIVERSARY'24.csv
- Копия Анализ аудитории - Ballmer Peak'24.csv
- Копия Анализ аудитории - CSFEST'25.csv
- Копия Анализ аудитории - GLANZ.csv
- Копия Анализ аудитории - Антипосвят'23.csv
- Копия Анализ аудитории - Антипосвят'25.csv
- Копия Анализ аудитории - Бал ФКН'24.csv
- Копия Анализ аудитории - ЗВ'25.csv
- Копия Анализ аудитории - ЗВ'26.csv
- Копия Анализ аудитории - Коллаб'24.csv
- Копия Анализ аудитории - Мероприятия.csv
- Копия Анализ аудитории - Настолки'24.csv
- Копия Анализ аудитории - Нейрорейв.csv
- Копия Анализ аудитории - Посвят'23.csv
- Копия Анализ аудитории - Посвят'25.csv
- Копия Анализ аудитории - Поход'24.csv
- Копия Анализ аудитории - Поход'25.csv
- Копия Анализ аудитории - Сводная.csv
- Копия Анализ аудитории - Экватор'24.csv
- Копия Анализ аудитории - Экватор'25.csv


In [82]:
def read_csv_robust(path: Path, nrows=None) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                dtype=str,
                encoding=encoding,
                sep=None,
                engine="python",
                nrows=nrows,
                on_bad_lines="skip",
            )
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать {path.name}: {last_error}")


def clean_filename_event_name(path: Path) -> str:
    name = path.stem
    prefix = "Копия Анализ аудитории - "

    if name.startswith(prefix):
        name = name[len(prefix):]

    return name.strip()


def normalize_colname(col: str) -> str:
    col = str(col).strip().lower().replace("ё", "е")
    col = re.sub(r"\s+", " ", col)
    return col

In [83]:
file_profiles = []
column_profiles = []
load_errors = []
dataframes = {}

for path in raw_files:
    event_name = clean_filename_event_name(path)

    try:
        df = read_csv_robust(path)
        dataframes[event_name] = df

        n_rows, n_cols = df.shape

        file_profiles.append({
            "source_file": path.name,
            "inferred_event_name": event_name,
            "rows": n_rows,
            "columns": n_cols,
            "file_size_kb": round(path.stat().st_size / 1024, 1),
            "empty_columns": int(df.isna().all().sum()),
            "duplicate_full_rows": int(df.duplicated().sum()) if n_rows > 0 else 0,
        })

        for col in df.columns:
            s = df[col]

            column_profiles.append({
                "source_file": path.name,
                "inferred_event_name": event_name,
                "column_name": col,
                "column_norm": normalize_colname(col),
                "non_null_count": int(s.notna().sum()),
                "null_count": int(s.isna().sum()),
                "null_share": round(float(s.isna().mean()), 4) if len(s) else None,
                "n_unique": int(s.nunique(dropna=True)),
            })

    except Exception as e:
        load_errors.append({
            "source_file": path.name,
            "error": str(e)
        })

file_profiles_df = pd.DataFrame(file_profiles)
column_profiles_df = pd.DataFrame(column_profiles)
load_errors_df = pd.DataFrame(load_errors)

file_profiles_df.sort_values("rows", ascending=False)

,source_file,inferred_event_name,rows,columns,file_size_kb,empty_columns,duplicate_full_rows
17,Копия Анализ аудитории - Сводная.csv,Сводная,5389,9,523.4,0,86
12,Копия Анализ аудитории - Нейрорейв.csv,Нейрорейв,859,4,98.0,0,4
6,Копия Анализ аудитории - Бал ФКН'24.csv,Бал ФКН'24,705,5,74.5,0,1
10,Копия Анализ аудитории - Мероприятия.csv,Мероприятия,644,93,328.9,37,95
14,Копия Анализ аудитории - Посвят'25.csv,Посвят'25,417,6,44.2,0,0
2,Копия Анализ аудитории - CSFEST'25.csv,CSFEST'25,371,7,40.3,0,0
13,Копия Анализ аудитории - Посвят'23.csv,Посвят'23,354,5,30.6,0,0
18,Копия Анализ аудитории - Экватор'24.csv,Экватор'24,316,6,19.3,0,45
16,Копия Анализ аудитории - Поход'25.csv,Поход'25,315,8,37.8,0,0
15,Копия Анализ аудитории - Поход'24.csv,Поход'24,287,8,35.5,0,0


In [84]:
load_errors_df

""


In [85]:
COLUMN_PATTERNS = {
    "pii_name": [
        r"\bфио\b", r"фамил", r"\bимя\b", r"отчеств", r"name", r"surname"
    ],
    "pii_telegram": [
        r"telegram", r"телеграм", r"\btg\b", r"ник", r"nickname", r"username", r"t\.me"
    ],
    "pii_phone": [
        r"телефон", r"номер", r"phone", r"mobile"
    ],
    "pii_email": [
        r"email", r"e-mail", r"почта", r"mail"
    ],
    "pii_ip": [
        r"ip"
    ],
    "registration_time": [
        r"дата отправки", r"отправлено", r"время", r"timestamp", r"registered", r"created"
    ],
    "payment": [
        r"оплат", r"payment", r"paid", r"цена", r"стоим", r"price", r"сумма", r"билет"
    ],
    "attendance": [
        r"пришел", r"пришла", r"посещ", r"attendance", r"attended", r"check", r"явка"
    ],
    "course": [
        r"курс", r"year"
    ],
    "group": [
        r"группа", r"group"
    ],
    "program": [
        r"направ", r"программа", r"program", r"faculty", r"факультет"
    ],
    "audience_type": [
        r"статус", r"критер", r"роль", r"аудитор", r"тип участ"
    ],
    "company_or_team": [
        r"команд", r"company", r"компан", r"пара", r"partner", r"тим"
    ],
    "free_text": [
        r"коммент", r"пожел", r"вопрос", r"отзыв", r"comment", r"note"
    ],
}


def detect_column_roles(column_name: str) -> list[str]:
    c = normalize_colname(column_name)
    roles = []

    for role, patterns in COLUMN_PATTERNS.items():
        if any(re.search(pattern, c) for pattern in patterns):
            roles.append(role)

    return roles or ["unknown"]


column_profiles_df["detected_roles"] = column_profiles_df["column_name"].apply(
    lambda x: ", ".join(detect_column_roles(x))
)

column_profiles_df["is_pii_candidate"] = column_profiles_df["detected_roles"].str.contains("pii_")

column_profiles_df.head(30)

,source_file,inferred_event_name,column_name,column_norm,non_null_count,null_count,null_share,n_unique,detected_roles,is_pii_candidate
0,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,ФИО,фио,86,0,0.0000,86,pii_name,True
1,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,Телеграм,телеграм,83,3,0.0349,78,pii_telegram,True
2,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,Критерий,критерий,86,0,0.0000,3,audience_type,False
3,Копия Анализ аудитории - ANNIVERSARY'24.csv,ANNIVERSARY'24,Оплачено,оплачено,86,0,0.0000,2,unknown,False
4,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,Название команды,название команды,51,99,0.6600,51,company_or_team,False
5,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,Участники,участники,148,2,0.0133,145,pii_telegram,True
6,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,Tg @,tg @,148,2,0.0133,144,pii_telegram,True
7,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,ВУЗ,вуз,148,2,0.0133,8,unknown,False
8,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,Направление,направление,148,2,0.0133,10,program,False
9,Копия Анализ аудитории - Ballmer Peak'24.csv,Ballmer Peak'24,Всем 18?,всем 18?,50,100,0.6667,2,unknown,False


In [86]:
PII_ROLES = ["pii_name", "pii_telegram", "pii_phone", "pii_email", "pii_ip"]

pii_coverage = []

for event_name, group in column_profiles_df.groupby("inferred_event_name"):
    roles = set()

    for role_str in group["detected_roles"]:
        roles.update(role_str.split(", "))

    row = {"inferred_event_name": event_name}

    for role in PII_ROLES:
        row[role] = role in roles

    row["strong_id_available"] = any(
        row[r] for r in ["pii_telegram", "pii_phone", "pii_email"]
    )

    if row["strong_id_available"]:
        row["stitching_potential"] = "high"
    elif row["pii_name"]:
        row["stitching_potential"] = "medium"
    else:
        row["stitching_potential"] = "low"

    pii_coverage.append(row)

pii_coverage_df = pd.DataFrame(pii_coverage)

pii_coverage_df.sort_values(["stitching_potential", "inferred_event_name"])

,inferred_event_name,pii_name,pii_telegram,pii_phone,pii_email,pii_ip,strong_id_available,stitching_potential
0,ANNIVERSARY'24,True,True,False,False,False,True,high
1,Ballmer Peak'24,True,True,False,False,False,True,high
2,CSFEST'25,True,True,False,True,False,True,high
3,GLANZ,True,True,False,False,False,True,high
4,Антипосвят'23,True,True,True,False,False,True,high
5,Антипосвят'25,True,True,True,False,False,True,high
7,ЗВ'25,True,True,True,False,False,True,high
8,ЗВ'26,True,True,False,False,False,True,high
11,Настолки'24,True,True,False,False,False,True,high
12,Нейрорейв,True,False,False,True,False,True,high


In [87]:
from pathlib import Path

MANUAL_DIR = Path("/content/manual")
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

aliases_text = """raw_event_name,canonical_event_id,canonical_event_name,source,notes
Посвят'23,posvyat_2023,Посвят'23,raw/svodnaya,
Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,
Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,
ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,
Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,
Коллаб'24,collab_2024,Коллаб'24,raw/svodnaya,
Настолки'24,nastolki_2024,Настолки'24,raw/svodnaya,
Нейрорейв,neyrorave_2024,Нейрорейв,raw/svodnaya,
Поход'24,pohod_2024,Поход'24,raw/svodnaya,
Экватор'24,ekvator_2024,Экватор'24,raw/svodnaya,
GLANZ,glanz_2024,GLANZ,raw/svodnaya,
ЗВ'25,zv_2025,ЗВ'25,raw,Проверить соответствие со сводной
ЗВ'24,zv_2025,ЗВ'25,svodnaya,Возможный alias - нужно подтвердить
Посвят'25,posvyat_2025,Посвят'25,raw/svodnaya,
Антипосвят'25,antiposvyat_2025,Антипосвят'25,raw/svodnaya,
CSFEST'25,csfest_2025,CSFEST'25,raw,Есть raw-файл но отсутствует в Сводной
Поход'25,pohod_2025,Поход'25,raw/svodnaya,
Экватор'25,ekvator_2025,Экватор'25,raw/svodnaya,
ЗВ'26,zv_2026,ЗВ'26,raw/svodnaya,
"""

(MANUAL_DIR / "event_aliases.csv").write_text(aliases_text, encoding="utf-8")

print((MANUAL_DIR / "event_aliases.csv").read_text(encoding="utf-8"))

raw_event_name,canonical_event_id,canonical_event_name,source,notes
Посвят'23,posvyat_2023,Посвят'23,raw/svodnaya,
Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,
Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,
ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,
Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,
Коллаб'24,collab_2024,Коллаб'24,raw/svodnaya,
Настолки'24,nastolki_2024,Настолки'24,raw/svodnaya,
Нейрорейв,neyrorave_2024,Нейрорейв,raw/svodnaya,
Поход'24,pohod_2024,Поход'24,raw/svodnaya,
Экватор'24,ekvator_2024,Экватор'24,raw/svodnaya,
GLANZ,glanz_2024,GLANZ,raw/svodnaya,
ЗВ'25,zv_2025,ЗВ'25,raw,Проверить соответствие со сводной
ЗВ'24,zv_2025,ЗВ'25,svodnaya,Возможный alias - нужно подтвердить
Посвят'25,posvyat_2025,Посвят'25,raw/svodnaya,
Антипосвят'25,antiposvyat_2025,Антипосвят'25,raw/svodnaya,
CSFEST'25,csfest_2025,CSFEST'25,raw,Есть raw-файл но отсутствует в Сводной
Поход'25,pohod_2025,Поход'25,raw/svodnaya,
Экватор'25,ekvator_2025,Эква

In [88]:
def load_manual_csv(filename: str) -> pd.DataFrame:
    path = MANUAL_DIR / filename

    if not path.exists():
        print(f"Нет файла: {path}")
        return pd.DataFrame()

    return pd.read_csv(path, dtype=str)


metadata = load_manual_csv("event_metadata.csv")
aliases = load_manual_csv("event_aliases.csv")
attendance_corrections = load_manual_csv("attendance_corrections.csv")

print("metadata:", metadata.shape)
print("aliases:", aliases.shape)
print("attendance_corrections:", attendance_corrections.shape)

metadata.head()

metadata: (18, 13)
aliases: (19, 5)
attendance_corrections: (1, 7)


,event_id,event_name,event_family,event_year,event_date,event_season,event_type,is_paid,approx_capacity,target_audience,format,strategic_role,notes
0,posvyat_2023,Посвят'23,Посвят,2023,NaN,autumn,onboarding,NaN,300-500,first-year students,overnight trip,acquisition,Посвящение в студенты; выезд с ночевкой в Подмосковье; квесты и вечерняя программа
1,antiposvyat_2023,Антипосвят'23,Антипосвят,2023,NaN,autumn,party,NaN,150-200,senior students,offline party,retention/community,Мероприятие от первокурсников для старшекурсников
2,ball_fkn_2024,Бал ФКН'24,Бал ФКН,2024,NaN,spring,party,NaN,unknown,HSE FCS students,offline party,community,Весенний бал ФКН
3,anniversary_2024,ANNIVERSARY'24,Anniversary,2024,NaN,unknown,party,NaN,unknown,cstati community,offline party,community,Единоразовая тусовка в честь юбилея cstati
4,ballmer_peak_2024,Ballmer Peak'24,Ballmer Peak,2024,NaN,spring,hackathon,NaN,unknown,students,mini hackathon,engagement,Мини-хахатон


In [89]:
required_metadata_cols = [
    "event_id",
    "event_name",
    "event_family",
    "event_year",
    "event_date",
    "event_season",
    "event_type",
    "is_paid",
    "approx_capacity",
    "target_audience",
    "format",
    "strategic_role",
    "notes",
]

required_alias_cols = [
    "raw_event_name",
    "canonical_event_id",
    "canonical_event_name",
    "source",
    "notes",
]

required_correction_cols = [
    "event_id",
    "metric_name",
    "old_value",
    "new_value",
    "correction_type",
    "confidence",
    "notes",
]

checks = []

for file_name, df, required_cols in [
    ("event_metadata.csv", metadata, required_metadata_cols),
    ("event_aliases.csv", aliases, required_alias_cols),
    ("attendance_corrections.csv", attendance_corrections, required_correction_cols),
]:
    missing_cols = [c for c in required_cols if c not in df.columns]

    checks.append({
        "file": file_name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_required_columns": ", ".join(missing_cols),
        "is_ok": len(missing_cols) == 0 and len(df) > 0,
    })

manual_checks_df = pd.DataFrame(checks)

manual_checks_df

,file,rows,columns,missing_required_columns,is_ok
0,event_metadata.csv,18,13,,True
1,event_aliases.csv,19,5,,True
2,attendance_corrections.csv,1,7,,True


In [90]:
import unicodedata

AUXILIARY_SOURCE_NAMES = {
    "Сводная",
    "Мероприятия",
}


def normalize_event_name_for_match(x: str) -> str:
    if pd.isna(x):
        return ""

    x = str(x)
    x = unicodedata.normalize("NFC", x)
    x = x.strip().lower().replace("ё", "е")
    x = re.sub(r"\s+", " ", x)

    return x


raw_event_names = pd.DataFrame({
    "raw_event_name_detected": [clean_filename_event_name(p) for p in raw_files]
})

raw_event_names["is_auxiliary_source"] = raw_event_names["raw_event_name_detected"].isin(
    AUXILIARY_SOURCE_NAMES
)

raw_event_names["raw_event_name_norm"] = raw_event_names["raw_event_name_detected"].apply(
    normalize_event_name_for_match
)

aliases_fixed = aliases.copy()
aliases_fixed["raw_event_name_norm"] = aliases_fixed["raw_event_name"].apply(
    normalize_event_name_for_match
)

raw_alias_check = raw_event_names.merge(
    aliases_fixed,
    on="raw_event_name_norm",
    how="left",
)

raw_alias_check["has_alias"] = (
    raw_alias_check["canonical_event_id"].notna()
    | raw_alias_check["is_auxiliary_source"]
)

raw_alias_check["alias_status"] = np.where(
    raw_alias_check["is_auxiliary_source"],
    "auxiliary_source_not_event",
    np.where(raw_alias_check["canonical_event_id"].notna(), "matched", "missing_alias")
)

raw_alias_check.sort_values(["has_alias", "raw_event_name_detected"])

,raw_event_name_detected,is_auxiliary_source,raw_event_name_norm,raw_event_name,canonical_event_id,canonical_event_name,source,notes,has_alias,alias_status
0,ANNIVERSARY'24,False,anniversary'24,ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,NaN,True,matched
1,Ballmer Peak'24,False,ballmer peak'24,Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,NaN,True,matched
2,CSFEST'25,False,csfest'25,CSFEST'25,csfest_2025,CSFEST'25,raw,Есть raw-файл но отсутствует в Сводной,True,matched
3,GLANZ,False,glanz,GLANZ,glanz_2024,GLANZ,raw/svodnaya,NaN,True,matched
4,Антипосвят'23,False,антипосвят'23,Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,NaN,True,matched
5,Антипосвят'25,False,антипосвят'25,Антипосвят'25,antiposvyat_2025,Антипосвят'25,raw/svodnaya,NaN,True,matched
6,Бал ФКН'24,False,бал фкн'24,Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,NaN,True,matched
7,ЗВ'25,False,зв'25,ЗВ'25,zv_2025,ЗВ'25,raw,Проверить соответствие со сводной,True,matched
8,ЗВ'26,False,зв'26,ЗВ'26,zv_2026,ЗВ'26,raw/svodnaya,NaN,True,matched
9,Коллаб'24,False,коллаб'24,Коллаб'24,collab_2024,Коллаб'24,raw/svodnaya,NaN,True,matched


In [91]:
quality_notes = []

if not load_errors_df.empty:
    for _, row in load_errors_df.iterrows():
        quality_notes.append({
            "severity": "high",
            "area": "load_error",
            "item": row["source_file"],
            "note": row["error"],
        })

for _, row in file_profiles_df.iterrows():
    if row["duplicate_full_rows"] > 0:
        severity = "medium"

        if row["inferred_event_name"] in AUXILIARY_SOURCE_NAMES:
            severity = "low"

        quality_notes.append({
            "severity": severity,
            "area": "duplicates",
            "item": row["source_file"],
            "note": f"Найдено полных дублей строк: {row['duplicate_full_rows']}",
        })

    if row["empty_columns"] > 0:
        quality_notes.append({
            "severity": "low",
            "area": "empty_columns",
            "item": row["source_file"],
            "note": f"Полностью пустых колонок: {row['empty_columns']}",
        })

missing_event_aliases = raw_alias_check[
    raw_alias_check["alias_status"] == "missing_alias"
]

for _, row in missing_event_aliases.iterrows():
    quality_notes.append({
        "severity": "high",
        "area": "event_alias",
        "item": row["raw_event_name_detected"],
        "note": "Raw event не найден в event_aliases.csv",
    })

for _, row in attendance_corrections.iterrows():
    quality_notes.append({
        "severity": "medium",
        "area": "manual_correction",
        "item": row.get("event_id", ""),
        "note": row.get("notes", ""),
    })

quality_notes_df = pd.DataFrame(quality_notes)

quality_notes_df

,severity,area,item,note
0,low,empty_columns,Копия Анализ аудитории - Ballmer Peak'24.csv,Полностью пустых колонок: 2
1,medium,duplicates,Копия Анализ аудитории - GLANZ.csv,Найдено полных дублей строк: 112
2,medium,duplicates,Копия Анализ аудитории - Бал ФКН'24.csv,Найдено полных дублей строк: 1
3,medium,duplicates,Копия Анализ аудитории - ЗВ'25.csv,Найдено полных дублей строк: 1
4,low,empty_columns,Копия Анализ аудитории - Коллаб'24.csv,Полностью пустых колонок: 1
5,low,duplicates,Копия Анализ аудитории - Мероприятия.csv,Найдено полных дублей строк: 95
6,low,empty_columns,Копия Анализ аудитории - Мероприятия.csv,Полностью пустых колонок: 37
7,medium,duplicates,Копия Анализ аудитории - Нейрорейв.csv,Найдено полных дублей строк: 4
8,low,duplicates,Копия Анализ аудитории - Сводная.csv,Найдено полных дублей строк: 86
9,medium,duplicates,Копия Анализ аудитории - Экватор'24.csv,Найдено полных дублей строк: 45


In [92]:
audit_outputs = {
    "file_profiles.csv": file_profiles_df,
    "column_profiles.csv": column_profiles_df,
    "pii_coverage.csv": pii_coverage_df,
    "manual_checks.csv": manual_checks_df,
    "raw_alias_check.csv": raw_alias_check,
    "quality_notes.csv": quality_notes_df,
}

for filename, df in audit_outputs.items():
    if df is not None and not df.empty:
        output_path = INTERIM_AUDIT_DIR / filename
        df.to_csv(output_path, index=False)
        print("saved:", output_path)

saved: /content/data/interim_private/audit/file_profiles.csv
saved: /content/data/interim_private/audit/column_profiles.csv
saved: /content/data/interim_private/audit/pii_coverage.csv
saved: /content/data/interim_private/audit/manual_checks.csv
saved: /content/data/interim_private/audit/raw_alias_check.csv
saved: /content/data/interim_private/audit/quality_notes.csv


## Предварительные выводы data audit

После первого запуска ноутбука нужно кратко зафиксировать:

1. Какие файлы успешно прочитались.
2. Какие файлы требуют special loader.
3. Какие события не сопоставились через `event_aliases.csv`.
4. Где есть сильные идентификаторы для stitching.
5. Где есть проблемы качества данных.
6. Какие поля можно использовать для будущих продуктовых метрик.

Следующий шаг после этого ноутбука — `02_identity_resolution.ipynb`, где будет строиться логика participant stitching.

In [93]:
print("raw files:", len(raw_files))
print("loaded dataframes:", len(dataframes))
print("load errors:", len(load_errors_df))
print("manual checks ok:", manual_checks_df["is_ok"].all())

if "has_alias" in raw_alias_check.columns:
    print("aliases ok:", raw_alias_check["has_alias"].all())
    print("missing aliases:", raw_alias_check.loc[~raw_alias_check["has_alias"], "raw_event_name_detected"].tolist())

print("quality notes:", len(quality_notes_df))

display(file_profiles_df.sort_values("rows", ascending=False))
display(manual_checks_df)
display(raw_alias_check.sort_values(["has_alias", "raw_event_name_detected"]))
display(quality_notes_df)

raw files: 20
loaded dataframes: 20
load errors: 0
manual checks ok: True
aliases ok: True
missing aliases: []
quality notes: 11


,source_file,inferred_event_name,rows,columns,file_size_kb,empty_columns,duplicate_full_rows
17,Копия Анализ аудитории - Сводная.csv,Сводная,5389,9,523.4,0,86
12,Копия Анализ аудитории - Нейрорейв.csv,Нейрорейв,859,4,98.0,0,4
6,Копия Анализ аудитории - Бал ФКН'24.csv,Бал ФКН'24,705,5,74.5,0,1
10,Копия Анализ аудитории - Мероприятия.csv,Мероприятия,644,93,328.9,37,95
14,Копия Анализ аудитории - Посвят'25.csv,Посвят'25,417,6,44.2,0,0
2,Копия Анализ аудитории - CSFEST'25.csv,CSFEST'25,371,7,40.3,0,0
13,Копия Анализ аудитории - Посвят'23.csv,Посвят'23,354,5,30.6,0,0
18,Копия Анализ аудитории - Экватор'24.csv,Экватор'24,316,6,19.3,0,45
16,Копия Анализ аудитории - Поход'25.csv,Поход'25,315,8,37.8,0,0
15,Копия Анализ аудитории - Поход'24.csv,Поход'24,287,8,35.5,0,0


,file,rows,columns,missing_required_columns,is_ok
0,event_metadata.csv,18,13,,True
1,event_aliases.csv,19,5,,True
2,attendance_corrections.csv,1,7,,True


,raw_event_name_detected,is_auxiliary_source,raw_event_name_norm,raw_event_name,canonical_event_id,canonical_event_name,source,notes,has_alias,alias_status
0,ANNIVERSARY'24,False,anniversary'24,ANNIVERSARY'24,anniversary_2024,ANNIVERSARY'24,raw/svodnaya,NaN,True,matched
1,Ballmer Peak'24,False,ballmer peak'24,Ballmer Peak'24,ballmer_peak_2024,Ballmer Peak'24,raw/svodnaya,NaN,True,matched
2,CSFEST'25,False,csfest'25,CSFEST'25,csfest_2025,CSFEST'25,raw,Есть raw-файл но отсутствует в Сводной,True,matched
3,GLANZ,False,glanz,GLANZ,glanz_2024,GLANZ,raw/svodnaya,NaN,True,matched
4,Антипосвят'23,False,антипосвят'23,Антипосвят'23,antiposvyat_2023,Антипосвят'23,raw/svodnaya,NaN,True,matched
5,Антипосвят'25,False,антипосвят'25,Антипосвят'25,antiposvyat_2025,Антипосвят'25,raw/svodnaya,NaN,True,matched
6,Бал ФКН'24,False,бал фкн'24,Бал ФКН'24,ball_fkn_2024,Бал ФКН'24,raw/svodnaya,NaN,True,matched
7,ЗВ'25,False,зв'25,ЗВ'25,zv_2025,ЗВ'25,raw,Проверить соответствие со сводной,True,matched
8,ЗВ'26,False,зв'26,ЗВ'26,zv_2026,ЗВ'26,raw/svodnaya,NaN,True,matched
9,Коллаб'24,False,коллаб'24,Коллаб'24,collab_2024,Коллаб'24,raw/svodnaya,NaN,True,matched


,severity,area,item,note
0,low,empty_columns,Копия Анализ аудитории - Ballmer Peak'24.csv,Полностью пустых колонок: 2
1,medium,duplicates,Копия Анализ аудитории - GLANZ.csv,Найдено полных дублей строк: 112
2,medium,duplicates,Копия Анализ аудитории - Бал ФКН'24.csv,Найдено полных дублей строк: 1
3,medium,duplicates,Копия Анализ аудитории - ЗВ'25.csv,Найдено полных дублей строк: 1
4,low,empty_columns,Копия Анализ аудитории - Коллаб'24.csv,Полностью пустых колонок: 1
5,low,duplicates,Копия Анализ аудитории - Мероприятия.csv,Найдено полных дублей строк: 95
6,low,empty_columns,Копия Анализ аудитории - Мероприятия.csv,Полностью пустых колонок: 37
7,medium,duplicates,Копия Анализ аудитории - Нейрорейв.csv,Найдено полных дублей строк: 4
8,low,duplicates,Копия Анализ аудитории - Сводная.csv,Найдено полных дублей строк: 86
9,medium,duplicates,Копия Анализ аудитории - Экватор'24.csv,Найдено полных дублей строк: 45


## Итоги первичного data audit

Первичный аудит данных успешно завершён.

### Что проверено

- Найдено и прочитано 20 CSV-файлов.
- Все 20 файлов успешно загружены без ошибок чтения.
- Ручные справочники `event_metadata.csv`, `event_aliases.csv` и `attendance_corrections.csv` имеют корректную структуру.
- Все event-level raw-файлы сопоставлены с canonical event id.
- `Сводная.csv` и `Мероприятия.csv` определены как auxiliary sources, а не как отдельные события.
- Для `Нейрорейв` решена проблема сопоставления через нормализацию названий.

### Найденные data quality issues

В данных есть несколько ожидаемых проблем качества:

- полные дубли строк в `GLANZ`, `Экватор'24`, `Нейрорейв`, `Сводная`, `Мероприятия`;
- пустые колонки в `Ballmer Peak'24`, `Коллаб'24`, `Мероприятия`;
- известная ручная корректировка attendance для `Антипосвят'25`;
- `Мероприятия.csv` имеет wide/archive-структуру и не должен использоваться как primary source;
- `Сводная.csv` используется как backbone-источник, но не как единственная source of truth по всем полям.

### Решение по источникам

- `Сводная.csv` используется как основной event-participant backbone.
- Отдельные raw CSV по мероприятиям используются как приоритетный источник event-specific полей.
- `Мероприятия.csv` используется только как audit fallback.
- Для дальнейшей аналитики нужна отдельная логика deduplication и identity resolution.

### Следующий шаг

Следующий ноутбук — `02_identity_resolution.ipynb`.

Его цель — построить private participant key и связать участников между событиями через Telegram, телефон, email, ФИО и дополнительные fallback-признаки.